# Week 76, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w076d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from urllib.parse import urlsplit, urlencode
u = urlsplit("https://shop.example.com/search?" + urlencode({"q": "tea & cake", "page": 2}))
print(u.hostname, u.port, u.query)

**Predict before you run:** what will this print?



In [ ]:
import ipaddress
print(ipaddress.ip_address("10.0.5.9") in ipaddress.ip_network("10.0.0.0/16"), ipaddress.ip_network("10.0.0.0/16").num_addresses)

**Predict before you run:** what will this print?



In [ ]:
from http import HTTPStatus
print([f"{c} {HTTPStatus(c).phrase}" for c in (201, 304, 405, 422)])

**Predict before you run:** what will this print?



In [ ]:
import struct
header = struct.pack("!HI", 7, 1000)
print(len(header), header, struct.unpack("!HI", header))

**Predict before you run:** what will this print?



In [ ]:
def app(environ, start_response):
    start_response("200 OK", [("Content-Type", "text/plain")])
    return [b"Hi, ", environ["PATH_INFO"][1:].encode()]

seen = []
print(b"".join(app({"PATH_INFO": "/Ada"}, lambda s, h: seen.append(s))), seen)

### Exercise: Canonical URLs

Two URLs often name the same resource. Write `canonical(url)`: lower-case the scheme and host, drop the port if it's the scheme's default (80 for `http`, 443 for `https`), use `/` for an empty path, drop the fragment, and sort the query parameters by name (keeping the order of repeated values), re-encoding them with `urlencode`. Then `same_resource(a, b)`.

In [ ]:
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode

def canonical(url):
    ___

def same_resource(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_canonical():\n    \"normalising URLs\"\n    assert canonical(\"HTTPS://Shop.Example.com:443/search?q=tea&a=1&q=cake#top\") == \"https://shop.example.com/search?a=1&q=tea&q=cake\"\n    assert canonical(\"http://a.com\") == \"http://a.com/\" and canonical(\"http://a.com:8080/x\") == \"http://a.com:8080/x\"\n    assert same_resource(\"https://a.com/x?b=2&a=1\", \"https://A.com:443/x?a=1&b=2#frag\") and not same_resource(\"http://a.com/\", \"https://a.com/\")")

### Exercise: Messages from chunks

Write a generator `messages(chunks)`: `chunks` is an iterable of byte strings as they came from `recv`; yield each newline-terminated message as a `str` (decoded as UTF-8, without the newline), as soon as its newline has arrived, however the bytes were split, and raise `ValueError("unfinished message")` at the end if bytes are left over without a newline.

In [ ]:
def messages(chunks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_messages():\n    \"framing across chunks\"\n    chunks = [b\"HEL\", b\"LO\\nWOR\", b\"LD\\nna\", \"\u00efve\".encode()[:2], \"\u00efve\".encode()[2:] + b\"\\n\"]\n    assert list(messages(chunks)) == [\"HELLO\", \"WORLD\", \"na\u00efve\"]\n    gen = messages(iter([b\"one\\ntwo\\n\", b\"thr\"]))\n    assert next(gen) == \"one\" and next(gen) == \"two\"\n    try:\n        next(gen)\n        assert False\n    except ValueError as e:\n        assert str(e) == \"unfinished message\"")

### Exercise: Forge: a web framework

Write `HTTPError`, `abort`, `Request`, `Response`, `App`, `TestResponse` and `TestClient` as described above.

In [ ]:
import io, json, re
from http import HTTPStatus
from urllib.parse import parse_qs, urlsplit

class HTTPError(Exception):
    def __init__(self, status, message=None):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def make_app():\n    app = App()\n    notes = {1: {\"id\": 1, \"text\": \"buy tea\"}}\n    seen = []\n\n    @app.before\n    def require_key(request):\n        seen.append(request.path)\n        if request.path.startswith(\"/admin\") and request.headers.get(\"x-api-key\") != \"k\":\n            return Response.json({\"error\": \"no key\"}, 401)\n\n    @app.after\n    def add_header(request, response):\n        response.headers[\"X-Served-By\"] = \"atlas\"\n        return response\n\n    @app.route(\"/notes\")\n    def list_notes(request):\n        limit = int(request.query.get(\"limit\", \"100\"))\n        return [notes[k] for k in sorted(notes)][:limit]\n\n    @app.route(\"/notes\", methods=(\"POST\",))\n    def add_note(request):\n        data = request.json()\n        if not isinstance(data, dict) or not str(data.get(\"text\", \"\")).strip():\n            abort(422, \"text is required\")\n        new_id = max(notes, default=0) + 1\n        notes[new_id] = {\"id\": new_id, \"text\": data[\"text\"]}\n        return Response.json(notes[new_id], 201, {\"Location\": f\"/notes/{new_id}\"})\n\n    @app.route(\"/notes/<int:note_id>\", methods=(\"GET\", \"DELETE\"))\n    def one_note(request, note_id):\n        if note_id not in notes:\n            abort(404, f\"no note {note_id}\")\n        if request.method == \"DELETE\":\n            del notes[note_id]\n            return Response(status=204)\n        return notes[note_id]\n\n    @app.route(\"/hello/<name>\")\n    def hello(request, name):\n        return f\"Hello, {name}!\"\n\n    @app.route(\"/old\")\n    def old(request):\n        return Response.redirect(\"/notes\")\n\n    @app.route(\"/admin/stats\")\n    def stats(request):\n        return {\"notes\": len(notes)}\n\n    @app.route(\"/crash\")\n    def crash(request):\n        return 1 / 0\n\n    return app, seen\n\ndef test_routes():\n    \"routing, parameters, JSON and text\"\n    app, seen = make_app()\n    client = TestClient(app)\n    r = client.get(\"/notes\")\n    assert r.status_code == 200 and r.json() == [{\"id\": 1, \"text\": \"buy tea\"}] and r.headers[\"content-type\"] == \"application/json\"\n    assert r.headers[\"content-length\"] == str(len(r.body)) and r.headers[\"x-served-by\"] == \"atlas\"\n    r = client.get(\"/hello/Ada\")\n    assert r.status_code == 200 and r.text == \"Hello, Ada!\" and r.headers[\"content-type\"].startswith(\"text/plain\")\n    r = client.get(\"/old\")\n    assert r.status_code == 303 and r.headers[\"location\"] == \"/notes\" and r.body == b\"\"\n    assert seen == [\"/notes\", \"/hello/Ada\", \"/old\"]\n\ndef test_requests_and_errors():\n    \"bodies, queries, errors and hooks\"\n    app, _ = make_app()\n    client = TestClient(app)\n    r = client.post(\"/notes\", json={\"text\": \"call Bob\"})\n    assert r.status_code == 201 and r.headers[\"location\"] == \"/notes/2\" and r.json()[\"id\"] == 2\n    assert [n[\"id\"] for n in client.get(\"/notes?limit=1\").json()] == [1]\n    assert client.post(\"/notes\", json={\"text\": \"\"}).status_code == 422 and client.post(\"/notes\", json={\"text\": \"\"}).json() == {\"error\": \"text is required\"}\n    bad = client.request(\"POST\", \"/notes\", headers={\"Content-Type\": \"application/json\"})\n    assert bad.status_code == 422\n    assert client.get(\"/notes/9\").json() == {\"error\": \"no note 9\"} and client.get(\"/notes/9\").status_code == 404\n    assert client.delete(\"/notes/1\").status_code == 204 and client.get(\"/notes/1\").status_code == 404\n    r = client.request(\"PUT\", \"/notes/2\")\n    assert r.status_code == 405 and r.headers[\"allow\"] == \"DELETE, GET\"\n    assert client.get(\"/nowhere\").status_code == 404 and client.get(\"/crash\").json() == {\"error\": \"internal error\"}\n    assert client.get(\"/admin/stats\").status_code == 401 and client.get(\"/admin/stats\", headers={\"X-Api-Key\": \"k\"}).json() == {\"notes\": 1}\n    assert client.get(\"/crash\").headers[\"x-served-by\"] == \"atlas\"")

> **What a framework gives you.** Count how much of the Forge was about the **application** (the notes handlers in the checks) and how much was **framework**. The handlers are a few lines each, because the framework turned a WSGI environ into a `Request`, a dict into JSON, an `abort(404)` into a proper response, and a missing route into a 405 with the right `Allow` header. That's exactly the division of labour in Flask or Django, and why moving between them is easy once you know what's underneath. Notice too that you never opened a socket: the test client called the application directly, which is how framework test clients work.

### Exercise: Forge bonus: an HTTP cache

Write `caching(app, clock)`: WSGI middleware that caches successful (`200`) responses to `GET` requests that carry `Cache-Control: max-age=N` (and not `no-store` or `private`), keyed by path and query string. A later `GET` for the same key within `N` seconds (by `clock()`) is answered from the cache, without calling `app`, with an `Age` header (whole seconds since it was stored) added. If the request has `If-None-Match` equal to the cached (or fresh) response's `ETag`, answer `304 Not Modified` with the `ETag` and no body. Expired entries are dropped. Other methods pass straight through, and a successful `POST`, `PUT` or `DELETE` to a path **removes** every cached entry for that path. Also count `hits` and `misses` as attributes of the returned application.

In [ ]:
import re

def caching(app, clock):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cache():\n    \"max-age, Age, ETags and invalidation\"\n    import io\n    now = [0]\n    calls = []\n    def app(environ, start_response):\n        calls.append((environ[\"REQUEST_METHOD\"], environ[\"PATH_INFO\"]))\n        path = environ[\"PATH_INFO\"]\n        if path == \"/news\":\n            start_response(\"200 OK\", [(\"Content-Type\", \"text/plain\"), (\"Cache-Control\", \"max-age=60\"), (\"ETag\", '\"n1\"')])\n            return [b\"latest news\"]\n        if path == \"/private\":\n            start_response(\"200 OK\", [(\"Cache-Control\", \"private, max-age=60\")])\n            return [b\"yours\"]\n        start_response(\"204 No Content\", [])\n        return []\n    cached = caching(app, lambda: now[0])\n    def call(method, path, headers=None):\n        env = {\"REQUEST_METHOD\": method, \"PATH_INFO\": path, \"QUERY_STRING\": \"\", \"wsgi.input\": io.BytesIO(b\"\")}\n        for k, v in (headers or {}).items():\n            env[\"HTTP_\" + k.upper().replace(\"-\", \"_\")] = v\n        got = {}\n        body = b\"\".join(cached(env, lambda s, h, e=None: got.update(status=s, headers=dict(h))))\n        return got[\"status\"], got[\"headers\"], body\n    assert call(\"GET\", \"/news\")[2] == b\"latest news\" and len(calls) == 1\n    now[0] = 30\n    status, headers, body = call(\"GET\", \"/news\")\n    assert status == \"200 OK\" and body == b\"latest news\" and headers[\"Age\"] == \"30\" and len(calls) == 1\n    status, headers, body = call(\"GET\", \"/news\", {\"If-None-Match\": '\"n1\"'})\n    assert status == \"304 Not Modified\" and body == b\"\" and headers[\"ETag\"] == '\"n1\"' and len(calls) == 1\n    now[0] = 61\n    call(\"GET\", \"/news\")\n    assert len(calls) == 2\n    call(\"GET\", \"/private\")\n    call(\"GET\", \"/private\")\n    assert calls.count((\"GET\", \"/private\")) == 2\n    call(\"POST\", \"/news\")\n    call(\"GET\", \"/news\")\n    assert calls.count((\"GET\", \"/news\")) == 3\n    assert cached.hits == 2 and cached.misses == 5")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## Forge: a web framework

Build a small WSGI web framework. The pieces, all tested together through a test client:

- **`HTTPError(status, message=None)`** (the message defaults to the status's phrase) and **`abort(status, message=None)`**, which raises it.
- **`Request(environ)`**: `method` (upper case), `path`, `query` (a dict of the **first** value of each query parameter), `headers` (lower-case names with `-`, built from the `HTTP_...` keys, plus `content-type` from `CONTENT_TYPE`), `body` (the `CONTENT_LENGTH` bytes from `wsgi.input`), `params` (the route's path parameters), and `json()` (the decoded body; an empty body is `None`; invalid JSON aborts with `400`).
- **`Response(body=b"", status=200, headers=None, content_type="text/plain; charset=utf-8")`**: a `str` body is UTF-8 encoded, and a `Content-Type` is added if there's a body and no type was given; class methods **`Response.json(data, status=200, headers=None)`** and **`Response.redirect(location, status=303)`**.
- **`App`** with:
  - `@app.route(pattern, methods=("GET",))`, where patterns use `<name>` and `<int:name>` as on Day 5, and handlers are called as `handler(request, **params)`;
  - `@app.before` (hooks called with the request; if one returns a response, it's used without routing) and `@app.after` (hooks called with the request and response, returning the response to use);
  - `handle(request)`: route and call the handler, turning a returned dict or list into `Response.json`, a `str` or `bytes` into `Response`; `405` with an `Allow` header (sorted, `", "`-separated) for a path matched with the wrong method; an `HTTPError` into `Response.json({"error": message}, status)` (so a missing route gives `{"error": "Not Found"}` with `404`); any other exception into `{"error": "internal error"}` with `500`; after hooks run in every case;
  - `__call__(environ, start_response)`, making it a WSGI application, adding `Content-Length` (except for 204 and 304).
- **`TestClient(app)`** with `request(method, path, json_body=None, headers=None)` (the path may include a query string; a JSON body sets `CONTENT_TYPE`; headers become `HTTP_...` keys) and shortcuts `get`, `post(path, json=None)` and `delete`, each returning a `TestResponse` with `status_code`, `headers` (a dict with lower-case names), `body`, `text` and `json()`.

## Forge bonus: an HTTP cache

## Weekly quiz

## Week 76 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can describe the internet's layers, work with addresses and networks, resolve names through DNS, and handle URLs safely. ([Day 1](w076d1.html))
- I can read and write HTTP messages, choose methods and status codes, and use caching, cookies and redirects. ([Day 2](w076d2.html))
- I can call JSON APIs with timeouts, tokens from the environment, pagination and safe retries. ([Day 3](w076d3.html))
- I can use sockets correctly, with partial sends and reads, and frame messages with delimiters and length prefixes. ([Day 4](w076d4.html))
- I can route requests, write WSGI applications and middleware, serve files safely, and serve a keep-alive connection. ([Day 5](w076d5.html))
- I built a web framework with a test client, and an HTTP cache. (Today)

Tomorrow is a **rest day**. Next week is **web backends**: Flask and FastAPI themselves, REST design, authentication, and deploying a backend to the internet.